# 15 — Execution Module Quickstart

Order lifecycle, entry fills, and execution guidance for screener output. See the [execution README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/execution/README.md) for full documentation.

In [ ]:
from __future__ import annotations

from swing_screener.execution import Order, fill_entry_order
from swing_screener.execution.guidance import add_execution_guidance, ExecutionConfig

import pandas as pd

## Order Lifecycle

Create a pending entry order, then fill it to produce a position with auto-linked stop/take-profit orders.

In [ ]:
order = Order(
    order_id="ORD-001", ticker="AAPL", status="pending",
    order_type="BUY_LIMIT", quantity=100, limit_price=175.00,
    order_date="2024-01-15",
)
print(order)

In [ ]:
new_orders, new_positions = fill_entry_order(
    orders=[order], positions=[],
    order_id="ORD-001",
    fill_price=175.50,
    fill_date="2024-01-16",
    quantity=100,
    stop_price=170.00,
    tp_price=185.00,
)
print(f"New orders: {len(new_orders)}, New positions: {len(new_positions)}")

In [ ]:
print("Orders:")
for o in new_orders:
    print(f"  {o.order_id}: {o.status} {o.order_type} {o.ticker} (kind={o.order_kind})")

print("\nPositions:")
for p in new_positions:
    print(f"  {p.position_id}: {p.ticker} {p.shares} sh @ ${p.entry_price:.2f}, stop=${p.stop_price:.2f}")

## Execution Guidance on Screener Output

Generate suggested order types, prices, and bands from a synthetic screener DataFrame.

In [ ]:
data = {
    "ticker": ["AAPL", "NVDA", "MSFT"],
    "signal": ["breakout", "breakout", "pullback"],
    "last": [178.50, 485.00, 375.00],
    "breakout_level": [177.00, 480.00, 370.00],
    "ma20_level": [175.00, 470.00, 365.00],
    "atr14": [3.5, 12.0, 5.0],
}
df = pd.DataFrame(data)
df

In [ ]:
cfg = ExecutionConfig(breakout_stop_buffer_pct=0.002, pullback_atr_fraction=0.25)
result = add_execution_guidance(df, cfg)
result[[
    "ticker", "signal",
    "suggested_order_type", "suggested_order_price",
    "order_price_band_low", "order_price_band_high",
    "execution_note",
]]